# NumPy arrays: data, shape, and selection

**Live notebook · approximately 45 minutes**

- One small scientific dataset throughout
- Code and results stay visible
- Explanations and caveats belong in the spoken narrative

## Outcomes

After this block, you should be able to:

- interpret an array's shape, axes, and data type;
- select data with indices, slices, and Boolean conditions;
- distinguish common views from copies; and
- choose an appropriate way to create a small array.

## Setup

- Use the repository's default Pixi environment.
- Load the precomputed temperature grid; its generator is not part of this block.

In [1]:
import numpy as np

temperature = np.load("temperature_grid.npy", allow_pickle=False)
np.__version__, temperature.shape

('2.4.6', (2, 3, 4))

## From Python values to an array

- Python lists are general-purpose containers.
- NumPy arrays are homogeneous and multidimensional.
- The same operator can have different meanings.

In [2]:
readings_list = [1.5, 2.0, 2.5]
readings = np.array(readings_list)

readings_list * 2, readings * 2

([1.5, 2.0, 2.5, 1.5, 2.0, 2.5], array([3., 4., 5.]))

## A small scientific array

Temperature on a small grid:

- axis 0: measurement time;
- axis 1: north–south position, $y$;
- axis 2: east–west position, $x$.

In [3]:
times = ("06:00", "12:00")

temperature

array([[[18.2, 18.4, 18.7, 19. ],
        [17.8, 18. , 18.2, 18.5],
        [17.3, 17.6, 17.8, 18. ]],

       [[21.1, 21.3, 21.6, 21.8],
        [20.6, 20.9, 21.2, 21.4],
        [20.2, 20.4, 20.7, 21. ]]])

Shape describes the axes; data type describes each stored value.

In [4]:
{
    "shape": temperature.shape,
    "dimensions": temperature.ndim,
    "values": temperature.size,
    "data type": temperature.dtype,
    "stored bytes": temperature.nbytes,
}

{'shape': (2, 3, 4),
 'dimensions': 3,
 'values': 24,
 'data type': dtype('float64'),
 'stored bytes': 192}

## Create arrays deliberately

- Known values: `np.array`
- Regular coordinates: `np.arange` or `np.linspace`
- Known shape and initial value: `np.zeros`, `np.ones`, or `*_like`
- `np.empty`: only when every element will be overwritten

In [5]:
x = np.linspace(0.0, 3.0, num=4)
y = np.arange(3)
correction = np.zeros_like(temperature)

x, y, correction.shape

(array([0., 1., 2., 3.]), array([0, 1, 2]), (2, 3, 4))

## Select data by meaning

- An integer index removes an axis.
- A slice retains an axis.
- `:` means every position along an axis.

In [6]:
snapshot = temperature[0]
one_snapshot = temperature[0:1]

snapshot.shape, one_snapshot.shape

((3, 4), (1, 3, 4))

In [7]:
sensor_series = temperature[:, 1, 2]

times, sensor_series

(('06:00', '12:00'), array([18.2, 21.2]))

In [8]:
western_region = temperature[:, 1:, :2]

western_region.shape, western_region

((2, 2, 2),
 array([[[17.8, 18. ],
         [17.3, 17.6]],
 
        [[20.6, 20.9],
         [20.2, 20.4]]]))

### Boolean selection

- A comparison creates a Boolean mask.
- Selecting with the mask returns matching values as a new one-dimensional array.

In [9]:
warm_mask = temperature >= 20.0
warm_values = temperature[warm_mask]

warm_mask.shape, warm_mask.dtype, warm_values

((2, 3, 4),
 dtype('bool'),
 array([21.1, 21.3, 21.6, 21.8, 20.6, 20.9, 21.2, 21.4, 20.2, 20.4, 20.7,
        21. ]))

## Views and copies

- Basic slicing usually returns a view.
- Advanced and Boolean indexing return copies.
- Use `np.shares_memory` when the distinction matters.

In [10]:
working = temperature.copy()
view = working[:, :, :2]
view[...] += 100.0

np.shares_memory(working, view), working[0]

(True,
 array([[118.2, 118.4,  18.7,  19. ],
        [117.8, 118. ,  18.2,  18.5],
        [117.3, 117.6,  17.8,  18. ]]))

In [11]:
working = temperature.copy()
selected = working[[0, 1]]
selected[0, 0, 0] = -99.0

np.shares_memory(working, selected), working[0, 0, 0], selected[0, 0, 0]

(False, np.float64(18.2), np.float64(-99.0))

### Reshaping is not a copy guarantee

- Reshaping changes how the same values are interpreted.
- It may return a view or require a copy.
- Request `.copy()` when independent storage is required.

In [12]:
flat = temperature.reshape(-1)
isolated = temperature.reshape(-1).copy()

flat.shape, np.shares_memory(temperature, flat), np.shares_memory(temperature, isolated)

((24,), True, False)

## Predict before running

For `probe[:, 1:, ::2]`:

- What is the result shape?
- Is it a view or a copy?
- Will modifying it change `probe`?

In [13]:
probe = temperature.copy()
region = probe[:, 1:, ::2]
region[0, 0, 0] = -99.0

{
    "shape": region.shape,
    "shares memory": np.shares_memory(probe, region),
    "value in probe": probe[0, 1, 0],
}

{'shape': (2, 2, 2),
 'shares memory': True,
 'value in probe': np.float64(-99.0)}

For `probe[:, ::2, 1:]`:

- What is the result shape?
- Is it a view or a copy?
- Will modifying it change `probe`?

In [14]:
probe = temperature.copy()
region = probe[:, ::2, 1:]
region[0, 0, 0] = -99.0

{
    "shape": region.shape,
    "shares memory": np.shares_memory(probe, region),
    "value in probe": probe[0, 0, 1],
}

{'shape': (2, 2, 3),
 'shares memory': True,
 'value in probe': np.float64(-99.0)}

For `probe[[0, 1], ::2, 1:]`:

- What is the result shape?
- Is it a view or a copy?
- Will modifying it change `probe`?

In [15]:
probe = temperature.copy()
region = probe[[0, 1], ::2, 1:]
region[0, 0, 0] = -99.0

{
    "shape": region.shape,
    "shares memory": np.shares_memory(probe, region),
    "value in probe": probe[0, 0, 1],
}

{'shape': (2, 2, 3),
 'shares memory': False,
 'value in probe': np.float64(18.4)}

## Decision check

Before manipulating an array, ask:

1. What does each axis mean?
2. Is the shape what I expect?
3. Is the data type suitable?
4. Does this selection retain or remove an axis?
5. Do I need shared or independent storage?

## Takeaways

- Shape gives structure to values.
- Axes should have explicit scientific meaning.
- Indexing changes both values and shape.
- Views are efficient but can create surprising mutation.
- Copies provide isolation at a memory cost.